# Evaluate Cosmos3-Nano locally

**Validate labeled assets → run five VQA predictions → score locally.** Full eight-task inference is opt-in. Existing predictions can be scored without a model endpoint or GPU.

This notebook uses Cosmos Framework's native VLMEvalKit datasets and metrics. It does not upload outputs or prepare an external scoring submission. A labeled, fingerprinted scoring bundle is required. Dataset access does **not** grant permission to redistribute labels or media. Public release remains gated on approved data licensing/distribution and a named reference result.

The serving configuration targets the reasoner in `nvidia/Cosmos3-Nano` at `e59a53c25979a090fa8706c9acc0c254a6e89b92`.

## 1 · Configure

`SCORING_ASSETS_DIR` is mounted read-only by Compose. Prepare the bundle outside every source checkout before setup; it must contain `scoring-assets.json`, canonical tables, original labels, and media. DVC also requires its pinned local roberta-large metric model. `LAUNCHABLE_STATE_DIR` must likewise be outside the checkout and separate from the bundle. See the README for preparation. No AWS credentials are mounted.

For existing predictions, set `SCORE_ONLY = True`, `PREDICTIONS_FILE` to a native TSV/XLSX/JSON table, and `TASK` appropriately. Native identity columns must be retained. Set `ALLOW_SUBSET = True` only when intentionally scoring a partial dataset; coverage is recorded explicitly.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone
from IPython.display import FileLink, display

SCORE_ONLY = os.environ.get('EVAL_SCORE_ONLY') == '1'
PREDICTIONS_FILE = None
TASK = 'VANTAGE_VQA'
ALLOW_SUBSET = False
RUN_FULL = False
ASSETS = Path(os.environ.get('SCORING_ASSETS_DIR', '/scoring-assets'))
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SMOKE_WORK = Path('/results') / f'local-vqa-smoke-{STAMP}'
FULL_WORK = Path('/results') / f'local-full-{STAMP}'
SCORE_DIR = Path('/results') / f'local-score-only-{STAMP}'
CLI = [sys.executable, str(Path('launchable.py').resolve())]

def invoke(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end='')
    result.check_returncode()

if SCORE_ONLY and not PREDICTIONS_FILE:
    raise ValueError('Set PREDICTIONS_FILE to an existing native prediction table before score-only execution.')
print('Existing-prediction scoring' if SCORE_ONLY else 'Five-question inference and local scoring')
print('Assets:', ASSETS)

## 2 · Validate local labels and media

Missing labels, altered annotation fingerprints, or unavailable media fail before contacting the model. Public inference-only inputs cannot substitute for ground truth.

In [ ]:
selected = TASK if SCORE_ONLY else ('all' if RUN_FULL else 'VANTAGE_VQA')
invoke(CLI + ['prepare', '--assets-dir', str(ASSETS), '--benchmarks', selected])

## 3 · Connect only when generating predictions

For BYOM, set the complete `COSMOS_API_BASE` chat-completions URL and `COSMOS_MODEL` in the evaluator environment before running the notebook. Keep any API key in the environment, not notebook source. Score-only mode skips this step entirely.

In [ ]:
if not SCORE_ONLY:
    invoke(CLI + ['check', '--wait', '1800'])

## 4 · Generate and score, or score existing predictions

The default smoke runs five deterministically selected VQA examples and then the native local evaluator. It is a **subset**, not a full benchmark or published-score reproduction. Score-only invokes the native scoring CLI directly and never constructs an inference model.

In [ ]:
if SCORE_ONLY:
    command = [sys.executable, '-m', 'vlmeval.local_scoring', '--task', TASK,
               '--predictions', str(PREDICTIONS_FILE), '--assets-dir', str(ASSETS),
               '--output-dir', str(SCORE_DIR)]
    if ALLOW_SUBSET:
        command.append('--allow-subset')
    invoke(command)
    SUMMARY = SCORE_DIR / 'summary.json'
else:
    invoke(CLI + ['run', '--assets-dir', str(ASSETS), '--benchmarks', 'VANTAGE_VQA',
                  '--mode', 'all', '--smoke', '--work-dir', str(SMOKE_WORK)])
    SUMMARY = SMOKE_WORK / 'local-scores' / 'bench_00' / 'summary.json'

## 5 · Inspect the local score and provenance

Each summary records the task's native metric, explicit units, aggregation rule, row coverage, asset and prediction hashes, and scorer-source hashes. DVC includes metric-model fingerprints. There is no invented cross-task overall score or claim of official leaderboard parity. All result files are private and can contain answer-bearing diagnostics; review before sharing.

In [ ]:
summary = json.loads(SUMMARY.read_text())
print(json.dumps({key: summary[key] for key in ('task', 'status', 'coverage', 'primary_metric')}, indent=2))
display(FileLink(str(SUMMARY), url_prefix='/files'))

## 6 · Optional full eight-task evaluation

Only after every requested task validates, set `RUN_FULL = True` in configuration and rerun from validation. This generates all eight tasks and scores them locally in a fresh directory. It can take substantially longer than the smoke. The labeled bundle must cover every task, including the real local DVC metric model. Astro preserves original labels and native filtering and denominators, with annotation diagnostics recorded privately. A full run does not by itself establish official parity: native per-task results require an agreed release protocol and named reference result.

In [ ]:
if RUN_FULL and not SCORE_ONLY:
    invoke(CLI + ['run', '--assets-dir', str(ASSETS), '--benchmarks', 'all',
                  '--mode', 'all', '--work-dir', str(FULL_WORK)])
    invoke(CLI + ['results', '--work-dir', str(FULL_WORK)])
    display(FileLink(str(FULL_WORK / 'run.json'), url_prefix='/files'))
else:
    print('Full eight-task inference disabled.')

## Stop and retain results

From the launchable directory, `docker compose --profile local stop cosmos-endpoint` stops serving without deleting state. To restart an approved inference run, use `bash setup.sh` (or `bash setup.sh --brev` on Brev). `bash setup.sh --score-only` starts only the notebook for existing-prediction scoring. Stopping a container does not stop billing for the underlying Brev instance; stop that instance separately when finished.

Clear notebook outputs before sharing. Do not upload restricted labels, media, credentials, or prediction artifacts to the PR.